# Section 7 · Observability 1 — Console Tracing (no external services)

Strands emits OpenTelemetry (OTel) spans for every agent invocation, reasoning cycle, model call and tool call. The simplest way to see them: print them to the console.

> Each tracing notebook should run in a **fresh kernel**: OpenTelemetry allows only one global tracer provider per Python process.

In [ ]:
%pip install -q -r requirements.txt

In [ ]:
import os

MODEL_ID = os.getenv("BEDROCK_MODEL_ID", "global.anthropic.claude-sonnet-4-6")
REGION = os.getenv("AWS_REGION", os.getenv("AWS_DEFAULT_REGION", "us-east-1"))
os.environ.setdefault("AWS_REGION", REGION)

In [ ]:
from strands import Agent
from strands.models.bedrock import BedrockModel
from strands.telemetry import StrandsTelemetry
from strands_tools import calculator

# Print every span to stdout
strands_telemetry = StrandsTelemetry()
strands_telemetry.setup_console_exporter()

model = BedrockModel(model_id=MODEL_ID, region_name=REGION)
agent = Agent(model=model, system_prompt="You are a helpful assistant.", tools=[calculator], callback_handler=None)

response = agent("What is 15% of 2,847? Then explain in two sentences why tracing matters for AI agents.")
print("\n\nANSWER:\n", response)

In the output above, look for:
- an **`invoke_agent`** span (top level) with total token counts
- **`execute_event_loop_cycle`** spans — one per reasoning loop
- **`chat`** spans — each model call, with input/output tokens and latency
- **`execute_tool calculator`** — the tool call with its parameters and result